[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-09-review-and-patterns.ipynb#scrollTo=aa1b2c3d)

---
# Day 9 · Review and Patterns — Rebuild, Audit, and Cheat Sheet
**certified-journeys / ai-agents-certified** · Day 9 · Review

> **Goal for today:** Rebuild the Day 7 supervisor graph from memory without referencing previous notebooks, produce a one-page cheat sheet for the week's core patterns, and identify and fix the edge case where both subagents return empty results.


## Why Active Reconstruction Matters

There are two kinds of knowledge:
- **Recognition:** "I know this when I see it."
- **Reconstruction:** "I can produce this from scratch."

The capstone on Day 10 requires reconstruction. Copying from your Day 7 notebook during the capstone is the equivalent of looking at the answer key during an exam — you might produce the right output but you haven't built the mental model that lets you debug it when it breaks.

**Today's protocol:**
1. Close all previous notebooks.
2. Rebuild the supervisor graph from the description below — no peeking.
3. Run it and verify the output matches what Day 7 produced.
4. If you got stuck: note exactly where, then look it up, then re-do that section.

> **Tip:** The review day is not passive reading — rebuild the hardest graph from memory. What you can reconstruct from memory is what you actually understand; everything else is just familiarity.


## The Reconstruction Spec

Build a LangGraph supervisor graph that satisfies all of these constraints — without looking at Day 7:

1. **State schema** named `SupervisorState` with:
   - `messages` (append-only with `add_messages`)
   - `next` (routing string)
   - `research_notes` (optional string, researcher-owned)
   - `draft_text` (optional string, writer-owned)

2. **Router** using `with_structured_output` with a `Literal` type — no free-text parsing.

3. **Supervisor node** that writes only `next`, never touches `messages`.

4. **Two subagent nodes** (`researcher`, `writer`) that each write to their private key plus `messages`.

5. **Conditional edge** from supervisor using the `next` field; `"FINISH"` maps to `END`.

6. **Back-edges** from each subagent to supervisor.

7. **Entry point** = supervisor.

Start coding below before reading further.


In [ ]:
%pip install -q langgraph langchain-core langchain-openai pydantic


In [ ]:
# Rebuild the supervisor graph from memory
# Close Day 7 notebook before starting. Write every line yourself.

import os
from typing import Annotated, Literal, Optional, TypedDict
from unittest.mock import MagicMock
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

os.environ.setdefault("OPENAI_API_KEY", "sk-placeholder")

# ── 1. State schema ──────────────────────────────────────────────────────────
class SupervisorState(TypedDict):
    messages:       Annotated[list, add_messages]
    next:           str
    research_notes: Optional[str]
    draft_text:     Optional[str]

# ── 2. Typed router output ────────────────────────────────────────────────────
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]

MEMBERS = ["researcher", "writer"]

SUPERVISOR_SYS = f"""You coordinate agents: {MEMBERS}.
Choose who acts next, or FINISH when done. One word only."""

def make_supervisor_node(llm):
    router = llm.with_structured_output(RouteDecision)
    def node(state: SupervisorState) -> dict:
        msgs = [SystemMessage(content=SUPERVISOR_SYS)] + state["messages"]
        decision: RouteDecision = router.invoke(msgs)
        return {"next": decision.next}
    return node

# ── 3. Subagent nodes ─────────────────────────────────────────────────────────
def researcher(state: SupervisorState) -> dict:
    topic = state["messages"][-1].content[:60] if state["messages"] else "unknown"
    notes = f"Facts about {topic}:\n- Fact A\n- Fact B"
    return {
        "research_notes": notes,
        "messages": [AIMessage(content=f"[Researcher] Notes: {notes}", name="researcher")],
    }

def writer(state: SupervisorState) -> dict:
    notes = state.get("research_notes") or "No notes"
    draft = f"Summary:\n{notes}\n\n[Final paragraph]"
    return {
        "draft_text": draft,
        "messages": [AIMessage(content=f"[Writer] Draft: {draft[:60]}…", name="writer")],
    }

# ── 4. Graph assembly ─────────────────────────────────────────────────────────
def build_graph(llm):
    g = StateGraph(SupervisorState)
    g.add_node("supervisor", make_supervisor_node(llm))
    g.add_node("researcher", researcher)
    g.add_node("writer",     writer)
    g.add_conditional_edges(
        "supervisor",
        lambda s: s["next"],
        {"researcher": "researcher", "writer": "writer", "FINISH": END}
    )
    g.add_edge("researcher", "supervisor")
    g.add_edge("writer",     "supervisor")
    g.set_entry_point("supervisor")
    return g.compile()

# ── 5. Mock and run ───────────────────────────────────────────────────────────
mock_router = MagicMock()
mock_router.invoke.side_effect = [
    RouteDecision(next="researcher"),
    RouteDecision(next="writer"),
    RouteDecision(next="FINISH"),
]
mock_llm = MagicMock()
mock_llm.with_structured_output.return_value = mock_router

graph = build_graph(mock_llm)
print("Nodes:", list(graph.get_graph().nodes.keys()))

result = graph.invoke({
    "messages": [HumanMessage(content="What caused the 2008 financial crisis?")],
    "next": "", "research_notes": None, "draft_text": None,
})

print(f"\nFinal state:")
print(f"  research_notes: {result['research_notes'][:50]}…")
print(f"  draft_text:     {result['draft_text'][:50]}…")
print(f"  messages:       {len(result['messages'])} total")
print("\n✓ Reconstruction successful")


**What just happened?**
- You rebuilt the full supervisor graph: state schema → typed router → subagents → conditional edges → back-edges → entry point → compile.
- If anything surprised you or you had to look something up, that's valuable signal — note it in your Day 9 notes file.
- The reconstruction took fewer lines than Day 7 because you now understand the pattern, not just the mechanics.


## The LangGraph Patterns Cheat Sheet

This is your one-page reference for the five core patterns from this week.

### 1. State Reducers

| Reducer | Import | Behaviour |
|---------|--------|----------|
| `add_messages` | `from langgraph.graph.message import add_messages` | Appends new messages to list |
| `operator.add` | `import operator` | Appends to any list |
| *(none)* | — | **Last write wins** — default for all other fields |

Syntax: `field: Annotated[list, add_messages]`

### 2. Conditional Edge Routing

```python
graph.add_conditional_edges(
    source_node,
    lambda state: state["routing_field"],   # returns a string
    {"value_a": "node_a", "value_b": END}   # mapping: return value → destination
)
```

### 3. Human-in-the-Loop (HITL) Resumption

```python
# Compile with checkpointer and interrupt_before
graph = builder.compile(
    checkpointer=SqliteSaver.from_conn_string(":memory:"),
    interrupt_before=["human_review_node"],
)
# Run until interruption
graph.invoke(state, config={"configurable": {"thread_id": "t1"}})
# Resume with new input
graph.invoke({"messages": [HumanMessage(content="Approved.")]},
             config={"configurable": {"thread_id": "t1"}})
```

### 4. Checkpointer Setup

```python
# SQLite (local, persistent)
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver.from_conn_string("state.db")
graph = builder.compile(checkpointer=checkpointer)

# In-memory (testing)
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
```

### 5. Supervisor Handoff Pattern

```python
# State: next = routing field (only supervisor writes it)
# Supervisor node: returns {"next": "agent_name" | "FINISH"}
# All subagents: add_edge(subagent, "supervisor")
# Conditional edge: {"FINISH": END, ...agents...}
# Entry point: supervisor
```


## Edge Case Audit: What Happens When Both Agents Return Empty?

In the Day 7 graph, if the researcher returns `research_notes = ""` and the writer reads it and produces `draft_text = ""`, the supervisor might loop indefinitely — or produce a meaningless report.

**Three failure modes to test:**
1. Researcher returns empty `research_notes`
2. Writer reads empty `research_notes` and produces empty `draft_text`
3. Supervisor doesn't detect the empty state and routes back to researcher indefinitely


In [ ]:
# Audit: reproduce each failure mode and verify the fix
from typing import Optional

# ── Failure mode 1: researcher returns empty ──────────────────────────────────
def researcher_empty(state: SupervisorState) -> dict:
    """Simulates a researcher that finds nothing."""
    return {
        "research_notes": "",  # empty string, not None
        "messages": [AIMessage(content="[Researcher] No results found.", name="researcher")],
    }

def writer_empty_guard(state: SupervisorState) -> dict:
    """Writer that gracefully handles empty research notes."""
    notes = state.get("research_notes") or None
    if not notes:  # catches both None and empty string
        return {
            "draft_text": "INSUFFICIENT_DATA",  # sentinel, not empty string
            "messages": [AIMessage(
                content="[Writer] Cannot draft: no research notes available.",
                name="writer"
            )],
        }
    draft = f"Summary:\n{notes}"
    return {
        "draft_text": draft,
        "messages": [AIMessage(content=f"[Writer] Draft: {draft[:60]}…", name="writer")],
    }

# ── Failure mode 3: supervisor infinite loop guard ─────────────────────────────
class GuardedState(SupervisorState):
    pass  # Adding retry_count requires TypedDict extension — show pattern:

class SupervisorStateV2(TypedDict):
    messages:       Annotated[list, add_messages]
    next:           str
    research_notes: Optional[str]
    draft_text:     Optional[str]
    retry_count:    int

MAX_RETRIES = 2

GUARDED_SUPERVISOR_PROMPT = """
You coordinate agents: researcher, writer.
If the researcher returned empty notes and retry_count >= {max}, output FINISH.
Otherwise choose the next agent or FINISH. One word only.
""".format(max=MAX_RETRIES)

def researcher_with_counter(state: SupervisorStateV2) -> dict:
    """Returns empty and increments retry_count to allow supervisor to detect the loop."""
    current = state.get("retry_count", 0)
    return {
        "research_notes": "",
        "retry_count": current + 1,
        "messages": [AIMessage(content=f"[Researcher] Empty (attempt {current+1})", name="researcher")],
    }

# Verify the guard logic
test_state = SupervisorStateV2(
    messages=[HumanMessage(content="Test")],
    next="", research_notes="", draft_text=None, retry_count=0
)
r1 = researcher_with_counter(test_state)
print(f"After attempt 1: retry_count = {r1['retry_count']}")
test_state["retry_count"] = r1["retry_count"]

r2 = researcher_with_counter(test_state)
print(f"After attempt 2: retry_count = {r2['retry_count']}")
print(f"At retry_count = {r2['retry_count']}: supervisor should output FINISH (>= {MAX_RETRIES})")

# Verify writer sentinel
empty_result = writer_empty_guard({
    "messages": [], "next": "", "research_notes": "", "draft_text": None
})
print(f"\nWriter sentinel: draft_text = '{empty_result['draft_text']}'")
print("→ INSUFFICIENT_DATA is distinguishable from a real draft — consumer can check this")


**What just happened?**
- **`"" vs None`**: Empty string and None are both falsy in Python, but LangGraph state may distinguish them. Using `or None` normalises the check.
- **`INSUFFICIENT_DATA` sentinel**: Using a named sentinel string (not empty) means the downstream consumer — or the supervisor — can distinguish "writer ran but had nothing to work with" from "writer hasn't run yet".
- **`retry_count` in state** is the canonical loop-break mechanism: it's visible to the supervisor's LLM via the system prompt, so the model can decide to FINISH without hardcoding the condition in graph topology.


## LangGraph Platform Deployment Overview

After building locally, deploying to LangGraph Platform provides:

| Feature | Local | LangGraph Platform |
|---------|-------|-------------------|
| Checkpointing | SQLite file | Managed Postgres |
| Streaming | In-process | SSE over HTTP |
| Cron / scheduled runs | DIY | Built-in scheduler |
| Auth | None | API key + OAuth |
| Scaling | Single process | Horizontal workers |

**Deployment path:**
1. Define `langgraph.json` (graph entrypoint + dependencies)
2. `langgraph build` → Docker image
3. `langgraph deploy` → pushes to LangGraph Cloud
4. Call via REST: `POST /runs/stream` with `{input: ..., config: ...}`

For the capstone, we stay local with SQLite — the platform concepts translate directly.

**`langgraph.json` minimal example:**
```json
{
  "graphs": {
    "research_system": "./graph.py:graph"
  },
  "python_version": "3.11",
  "dependencies": [".", "langgraph", "langchain-openai"]
}
```


In [ ]:
# Self-audit: verify your reconstruction against the Day 7 specification

import inspect

checks = []

# Check 1: SupervisorState has the required fields
required_fields = {"messages", "next", "research_notes", "draft_text"}
actual_fields = set(SupervisorState.__annotations__.keys())
check1 = required_fields.issubset(actual_fields)
checks.append(("SupervisorState has all required fields", check1))

# Check 2: RouteDecision has Literal type with FINISH
route_annotation = RouteDecision.model_fields["next"].annotation
check2 = "FINISH" in str(route_annotation)
checks.append(("RouteDecision includes FINISH sentinel", check2))

# Check 3: graph has expected nodes
graph_nodes = set(graph.get_graph().nodes.keys())
expected_nodes = {"supervisor", "researcher", "writer", "__start__", "__end__"}
check3 = expected_nodes.issubset(graph_nodes)
checks.append(("Graph contains all required nodes", check3))

# Check 4: researcher and writer only write their own private fields
r_result = researcher({"messages": [HumanMessage(content="test")], "next": "", "research_notes": None, "draft_text": None})
check4 = "draft_text" not in r_result
checks.append(("researcher does not write draft_text", check4))

w_result = writer({"messages": [HumanMessage(content="test")], "next": "", "research_notes": "some notes", "draft_text": None})
check5 = "research_notes" not in w_result
checks.append(("writer does not write research_notes", check5))

# Check 5: empty-research guard produces INSUFFICIENT_DATA sentinel
sentinel_result = writer_empty_guard({"messages": [], "next": "", "research_notes": "", "draft_text": None})
check6 = sentinel_result["draft_text"] == "INSUFFICIENT_DATA"
checks.append(("Empty research guard produces INSUFFICIENT_DATA sentinel", check6))

print("Self-audit results:")
all_passed = True
for name, passed in checks:
    icon = "✓" if passed else "✗"
    print(f"  {icon} {name}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All checks passed — ready for the Day 10 capstone.")
else:
    print("Some checks failed — review the failing items before Day 10.")


**What just happened?**
- The self-audit programmatically verifies the reconstruction against the Day 7 specification — no manual eyeballing required.
- Each check tests a discrete architectural invariant: field presence, sentinel values, and private namespace discipline. These are the same checks the capstone grader will use.
- If any check fails, you know exactly which pattern to re-study before Day 10.


## Day 9 Pattern Reference: Quick Recall Table

Use this table for rapid review before the capstone. Cover the right column and try to recall each answer.

| Pattern / API | Detail to remember |
|---|---|
| `add_messages` import | `from langgraph.graph.message import add_messages` |
| Annotated reducer syntax | `field: Annotated[list, add_messages]` |
| Conditional edge signature | `add_conditional_edges(source, fn, {val: node})` |
| Terminate the graph | Map any value to `END` (from `langgraph.graph import END`) |
| HITL: pause before node | `compile(interrupt_before=["node_name"])` |
| HITL: resume | Same `thread_id` in config, pass new input |
| SQLite checkpointer | `SqliteSaver.from_conn_string("file.db")` |
| Supervisor loops subagents | `add_edge(subagent, "supervisor")` for each subagent |
| Prevent routing hallucinations | `llm.with_structured_output(TypedModel)` |
| Private namespace pattern | Each node returns only its own keys |
| Infinite loop guard | `retry_count` in state; check in supervisor prompt |
| Stream tokens | `graph.astream(state, stream_mode="messages")` |
| Stream node labels | `graph.astream(state, stream_mode="updates")` |
| Long-term memory | `InMemoryStore`, namespace = `(category, user_id)` |
| Inject store into node | `builder.compile(store=store)` |


In [ ]:
# Challenge: Rebuild the supervisor with the empty-result fix incorporated
#
# Build a complete supervisor graph (from scratch, no copy-paste) that:
# 1. Uses SupervisorStateV2 (with retry_count field).
# 2. Researcher increments retry_count on empty results, resets on success.
# 3. Writer uses the INSUFFICIENT_DATA sentinel when notes are empty.
# 4. Supervisor prompt instructs the LLM to FINISH if retry_count >= 2.
# 5. Run it twice:
#    a. Normal run: mock → researcher → writer → FINISH. Verify draft_text is not INSUFFICIENT_DATA.
#    b. Empty run: mock → researcher (empty) → researcher (empty) → FINISH.
#       Verify draft_text is INSUFFICIENT_DATA (writer was never invoked with good data).
#
# Scaffold:
# class SupervisorStateV2(TypedDict):  # already defined above
#     ...
#
# def researcher_v2(state): ...
# def writer_v2(state): ...
# def make_supervisor_v2(llm): ...
# def build_graph_v2(llm): ...
#
# Normal mock: [researcher, writer, FINISH]
# Empty mock:  [researcher, researcher, FINISH]

# Your solution here


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| Reconstruction vs recognition | Can you build it from scratch? That's the test |
| State reducers | `add_messages` appends; everything else is last-write-wins |
| Conditional edges | `add_conditional_edges(src, fn, mapping)` — fn returns a string |
| HITL resumption | Same `thread_id` + new input; graph resumes from interruption point |
| Checkpointer setup | `SqliteSaver.from_conn_string(path)` or `MemorySaver()` |
| Supervisor handoff | Supervisor writes `next`; subagents add_edge back to supervisor |
| Empty result guard | `retry_count` in state + sentinel value from writer |

> **Tip:** The review day is not passive reading — rebuild the hardest graph from memory. What you can reconstruct is what you actually understand.

---
## What's next
**Day 10** → Capstone — Build the full multi-agent research system with SQLite persistence, supervisor routing, DuckDuckGo search, summarizer, and node-level streaming.

Mark Day 9 complete in your [tracker](../index.html).
